# Tests

In [ ]:
import gc
import os
import sys
from pathlib import Path

os.environ.setdefault("PYTHONDONTWRITEBYTECODE", "1")

ROOT = Path.cwd()
SRC = ROOT / "src" / "local_learning"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import torch
from datasets import get_dataset
from models import get_model

DATASETS = (
    "cifar10",
    "cifar100",
    "cifar10_patches",
    "cifar100_patches",
)

ARCHITECTURES = (
    "cnn1",
    "cnn2",
    "resnet18",
    "pretrained_resnet18",
    "vgg16",
    "pretrained_vgg16",
    "greedy_stacked_autoencoder",
    "wta_conv_ae",
)

def base_cfg(architecture_type: str) -> dict:
    training_mode = "reconstruction" if architecture_type == "wta_conv_ae" else "classification"
    return {
        "architecture_type": architecture_type,
        "dataset": "cifar10",
        "training_mode": training_mode,
        "epochs": 1,
        "hidden_channels": 8,
        "num_layers": 2,
        "k_lifetime": 0.2,
        "k_spatial": None,
        "k_population": None,
        "local_training": False,
    }

dataset_results = []
model_results = []

## Datasets

In [ ]:
for dataset in DATASETS:
    for train in (True, False):
        ds = get_dataset(train=train, dataset=dataset, preprocessing="none")
        image, label = ds[0]
        result = {
            "dataset": dataset,
            "split": "train" if train else "test",
            "length": len(ds),
            "image_shape": tuple(image.shape),
            "label_shape": tuple(torch.as_tensor(label).shape),
        }
        dataset_results.append(result)
        print(result)

        del ds, image, label
        gc.collect()

## Models

In [ ]:
for architecture_type in ARCHITECTURES:
    model = get_model(base_cfg(architecture_type))
    total_params = sum(param.numel() for param in model.parameters())
    trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
    result = {
        "architecture_type": architecture_type,
        "class": type(model).__name__,
        "total_params": total_params,
        "trainable_params": trainable_params,
    }
    assert isinstance(model, torch.nn.Module)
    assert trainable_params > 0
    model_results.append(result)
    print(result)

    del model
    gc.collect()

## Summary

In [ ]:
print(f"Datasets checked: {len(dataset_results)}")
print(f"Models checked: {len(model_results)}")
assert len(dataset_results) == len(DATASETS) * 2
assert len(model_results) == len(ARCHITECTURES)